# Conversion Unimarc vers matrice excel

Import :
- le script SRUextractionMODIFIE.py (version Mathias) ou SRUextraction.py (version initiale) est nécessaire pour interroger le SRU BnF lorsqu'un ISNI n'est pas trouvé dans le mapping local ;
- le fichier Numelyo-mapping-isnis-nnas.txt doit être présent dans le répertoire de travail du notebook ;
- les lignes d'installation ci-dessous permettent de compléter les bibliothèques manquantes dans l'environnement Jupyter.


In [85]:
# Interrogation du SRU BnF : version modifiée si disponible, sinon version initiale
try:
    import SRUextractionMODIFIE as sru
except ModuleNotFoundError as e:
    if e.name != "SRUextractionMODIFIE":
        raise
    import SRUextraction as sru

try:
    import pandas
except ModuleNotFoundError:
    %pip install pandas
    import pandas

import re
from collections import Counter

try:
    from pymarc import MARCReader, Field
except ModuleNotFoundError:
    %pip install pymarc
    from pymarc import MARCReader, Field

try:
    from tqdm import tqdm
except ModuleNotFoundError:
    %pip install tqdm
    from tqdm import tqdm

try:
    from openpyxl.cell.cell import ILLEGAL_CHARACTERS_RE
except ModuleNotFoundError:
    %pip install openpyxl
    from openpyxl.cell.cell import ILLEGAL_CHARACTERS_RE


### Ajout du mapping ISNI-NNA
On cherche d'abord le NNA dans le fichier de correspondances fourni par la BML.
**Si l'ISNI n'est pas présent dans le mapping, ou si son NNA est vide, on interroge le SRU de la BnF.**
Les erreurs de requêtage sont conservées dans `ERREURS_SRU`, sans interrompre la conversion des autres notices.


In [86]:
import csv
from pathlib import Path

MAPPING_PATH = Path("Numelyo-mapping-isnis-nnas.txt")

# Les ISNI peuvent être saisis avec ou sans le préfixe "ISNI" dans les notices Unimarc
# (ex. 700$o0000000083509777 ou 700$oISNI0000000083509777).
def normaliser_isni(value):
    value = re.sub(r"^ISNI\s*:?\s*", "", str(value or "").strip(), flags=re.I)
    value = re.sub(r"[\s-]", "", value).upper()
    return value if re.fullmatch(r"\d{15}[\dX]", value) else ""

ISNI_TO_NNA = {}

with MAPPING_PATH.open(encoding="utf-8-sig", newline="") as f:
    reader = csv.DictReader(f, delimiter="\t")

    for row in reader:
        raw_isni = (row.get("ISNI") or "").strip().upper()
        nna = (row.get("NNA") or "").strip()

        # Certaines lignes ajoutent le nom de la collectivité après les 16 caractères de l'ISNI
        match = re.match(r"^(\d{15}[\dX])(?=\s|$)", raw_isni)
        if not match or not re.fullmatch(r"\d{8}", nna):
            continue

        isni = match.group(1)
        if isni in ISNI_TO_NNA and ISNI_TO_NNA[isni] != nna:
            raise ValueError(f"Correspondances NNA contradictoires pour l'ISNI {isni}")
        ISNI_TO_NNA[isni] = nna

# Le cache évite plusieurs interrogations SRU pour un même ISNI absent du mapping
SRU_CACHE_NNA = {}
ERREURS_SRU = {}

def recuperer_nna(isni):
    if not isni:
        return ""

    # Priorité au mapping BML, puis interrogation du SRU BnF en cas d'absence
    nna = ISNI_TO_NNA.get(isni, "")
    if nna:
        return nna
    if isni in SRU_CACHE_NNA:
        return SRU_CACHE_NNA[isni]

    nna = ""
    try:
        result = sru.SRU_result(
            f"aut.isni any '{isni}'",
            parametres={"recordSchema": "intermarcxchange"}
        )
        record_aut = result.firstRecord if result.test else None
        if record_aut is not None:
            value = sru.record2fieldvalue(record_aut, "001")
            value = str(value or "")

            # Dans les premiers scripts, le NNA est extrait de 001 par [5:].
            # Certaines formes possèdent en plus un caractère de contrôle final : [5:-1].
            candidat = value[5:]
            if re.fullmatch(r"\d{8}", candidat):
                nna = candidat
            elif re.fullmatch(r"\d{8}[0-9A-Za-z]", candidat):
                nna = candidat[:-1]
            elif value:
                ERREURS_SRU[isni] = f"Format 001 à contrôler : {value!r}"
    except Exception as e:
        # Garder la notice même si le service SRU n'est pas disponible
        ERREURS_SRU[isni] = str(e)

    SRU_CACHE_NNA[isni] = nna
    return nna

print(f"Mapping chargé : {len(ISNI_TO_NNA)} correspondances")


Mapping chargé : 1066 correspondances


### Référentiels
Ici, on commence par prendre les référentiels intermarc NG. 
Dans le cas des codes fonctions et des catégories techniques, les informations sont exprimées de façon différentes en Unimarc et en Intermarc NG : les dictionnaires ont pour but de permettre leur conversion.
Attention, ces dictionnaires ne sont pas exhaustifs : les seuls cas de figures pris en compte ici sont ceux explicitement discutés avec la BML. S'il y a utilisation d'autres codes fonctions ou d'autres informations en 200$b, la conversion ne sera pas satisfaisante.

In [87]:
CODES_PAYS = ["##", "ad", "ae", "af", "ag", "ai", "al", "am", "ao", "aq", "ar", "as", "at", "au", "aw", "ax", "az", "ba", "bb", "bd", "be", "bf", "bf", "bf", "bg", "bh", "bi", "bj", "bl", "bm", "bn", "bo", "bq", "bq", "bq", "br", "bs", "bt", "bv", "bw", "by", "by", "bz", "ca", "cc", "cc", "cd", "cd", "cf", "cf", "cf", "cg", "cg", "ch", "ch", "ci", "ck", "cl", "cm", "cn", "cn", "co", "cr", "cu", "cv", "cw", "cx", "cy", "cz", "cz", "de", "de", "dj", "dk", "dm", "do", "do", "dz", "ec", "ee", "eg", "eh", "er", "es", "et", "eu", "fi", "fj", "fk", "fk", "fm", "fm", "fo", "fr", "ga", "gb", "gb", "gd", "ge", "gf", "gg", "gh", "gi", "gl", "gm", "gn", "gp", "gq", "gr", "gs", "gt", "gu", "gw", "gy", "hk", "hm", "hn", "hr", "ht", "hu", "id", "ie", "il", "im", "im", "in", "io", "io", "io", "iq", "ir", "is", "it", "je", "jm", "jo", "jp", "ke", "kg", "kh", "ki", "km", "kn", "kn", "kp", "kp", "kr", "kr", "kw", "ky", "ky", "ky", "kz", "la", "lb", "lc", "li", "lk", "lr", "ls", "lt", "lu", "lv", "ly", "ly", "ma", "mc", "md", "md", "me", "mf", "mg", "mh", "mk", "ml", "mm", "mm", "mn", "mo", "mp", "mq", "mr", "ms", "mt", "mu", "mv", "mw", "mx", "my", "mz", "na", "nc", "ne", "nf", "ng", "ni", "nl", "no", "np", "nr", "nu", "nz", "om", "pa", "pe", "pf", "pg", "ph", "pk", "pl", "pm", "pn", "pr", "ps", "ps", "pt", "pw", "pw", "py", "qa", "re", "ro", "rs", "ru", "ru", "rw", "sa", "sb", "sc", "sd", "se", "sg", "sh", "sh", "sh", "si", "sj", "sk", "sl", "sm", "sn", "so", "sr", "ss", "ss", "st", "sv", "sv", "sx", "sx", "sy", "sz", "tc", "td", "tf", "tg", "th", "tj", "tk", "tl", "tm", "tn", "to", "tr", "tt", "tv", "tw", "tw", "tz", "ua", "ug", "um", "us", "uy", "uz", "va", "va", "vc", "ve", "vg", "vg", "vi", "vi", "vn", "vu", "wf", "ws", "ws", "xk", "ye", "ye", "yt", "za", "zm", "zw", "zz"]


In [88]:
CODES_LANGUE = ["###", "aar", "abk", "ace", "ace", "ach", "ach", "ach", "ada", "ada", "ada", "ada", "ady", "afa", "afa", "afh", "afr", "ain", "aka", "aka", "aka", "akk", "akk", "akk", "alb", "alb", "ale", "alg", "alg", "alg", "alg", "alg", "alg", "alg", "alg", "alg", "alg", "alg", "alg", "alg", "alt", "amh", "ang", "ang", "ang", "ang", "anp", "apa", "apa", "ara", "arc", "arc", "arc", "arg", "arm", "arn", "arn", "arn", "arp", "art", "art", "art", "arw", "asm", "ast", "ast", "ast", "ast", "ath", "ath", "ath", "ath", "ath", "ath", "ath", "ath", "ath", "ath", "ath", "ath", "ath", "ath", "ath", "ath", "ath", "ath", "ath", "ath", "ath", "ath", "ath", "ath", "ath", "ath", "aus", "aus", "aus", "aus", "aus", "aus", "aus", "ava", "ave", "ave", "ave", "ave", "ave", "ave", "awa", "aym", "aze", "aze", "bad", "bad", "bai", "bai", "bak", "bak", "bal", "bam", "bam", "ban", "baq", "baq", "baq", "bas", "bas", "bas", "bas", "bas", "bas", "bat", "bat", "bej", "bej", "bej", "bel", "bem", "ben", "ber", "ber", "ber", "ber", "ber", "ber", "ber", "ber", "ber", "ber", "ber", "ber", "ber", "ber", "bho", "bih", "bih", "bik", "bin", "bin", "bis", "bis", "bis", "bla", "bla", "bla", "bnt", "bnt", "bnt", "bnt", "bos", "bra", "bra", "bre", "bre", "bre", "bre", "bre", "bre", "btk", "btk", "bua", "bug", "bul", "bur", "byn", "byn", "byn", "cad", "cad", "cad", "cai", "cai", "car", "car", "car", "car", "cat", "cat", "cau", "cau", "cau", "cau", "cau", "cau", "ceb", "cel", "cel", "cel", "cel", "cel", "cel", "cha", "chb", "che", "chg", "chg", "chg", "chi", "chk", "chk", "chm", "chn", "chn", "chn", "chn", "cho", "chp", "chr", "chu", "chu", "chu", "chu", "chu", "chu", "chv", "chy", "cmc", "cmc", "cop", "cor", "cos", "cpe", "cpe", "cpe", "cpe", "cpe", "cpe", "cpe", "cpe", "cpf", "cpf", "cpf", "cpf", "cpf", "cpf", "cpf", "cpf", "cpf", "cpf", "cpp", "cpp", "cpp", "cpp", "cpp", "cre", "cre", "cre", "crh", "crh", "crp", "crp", "crp", "crp", "crp", "crp", "crp", "csb", "cus", "cus", "cus", "cus", "cze", "dak", "dak", "dak", "dak", "dak", "dak", "dak", "dan", "dar", "day", "day", "del", "den", "den", "dgr", "dgr", "dgr", "din", "din", "div", "doi", "dra", "dra", "dra", "dra", "dra", "dra", "dra", "dra", "dra", "dra", "dra", "dra", "dra", "dsb", "dua", "dum", "dum", "dum", "dut", "dut", "dyu", "dyu", "dyu", "dzo", "dzo", "efi", "efi", "egy", "egy", "egy", "egy", "eka", "elx", "elx", "elx", "elx", "elx", "elx", "elx", "eng", "enm", "enm", "enm", "epo", "est", "ewe", "ewe", "ewe", "ewo", "fan", "fan", "fao", "fat", "fij", "fil", "fil", "fin", "fin", "fin", "fiu", "fiu", "fiu", "fiu", "fiu", "fon", "frdbnf", "frdbnf", "fre", "frm", "frm", "frm", "fro", "fro", "fro", "fro", "frr", "frs", "fry", "ful", "ful", "ful", "ful", "ful", "fur", "gaa", "gay", "gba", "gedbnf", "gedbnf", "gem", "gem", "gem", "gem", "gem", "gem", "geo", "ger", "gez", "gez", "gez", "gez", "gez", "gez", "gil", "gil", "gla", "gla", "gla", "gla", "gle", "gle", "gle", "gle", "glg", "glv", "glv", "glv", "gmh", "gmh", "goh", "goh", "gon", "gon", "gor", "got", "grb", "grc", "grc", "grc", "grc", "grc", "grc", "grc", "grc", "grdbnf", "grdbnf", "gre", "gre", "grm", "grn", "grp", "gsw", "gsw", "gsw", "gsw", "gsw", "gsw", "guj", "guj", "guj", "gwi", "hai", "hat", "hat", "hat", "hau", "haw", "heb", "heb", "her", "hil", "him", "him", "him", "hin", "hit", "hmn", "hmo", "hrv", "hsb", "hsb", "hun", "hun", "hup", "iba", "ibo", "ibo", "ice", "ido", "iii", "ijo", "ijo", "ijo", "ijo", "iku", "ile", "ilo", "ina", "ina", "inc", "inc", "inc", "inc", "inc", "inc", "inc", "inc", "inc", "inc", "inc", "inc", "inc", "inc", "inc", "inc", "inc", "inc", "inc", "inc", "inc", "inc", "inc", "ind", "ind", "ine", "ine", "ine", "ine", "ine", "ine", "ine", "inh", "ipk", "ira", "ira", "ira", "ira", "ira", "ira", "ira", "ira", "iro", "iro", "iro", "iro", "iro", "iro", "iro", "iro", "ita", "itdbnf", "itdbnf", "itdbnf", "itdbnf", "jav", "jbo", "jpn", "jpr", "jrb", "kaa", "kab", "kab", "kac", "kac", "kal", "kal", "kam", "kan", "kan", "kan", "kan", "kar", "kar", "kas", "kas", "kas", "kas", "kau", "kaw", "kaw", "kaw", "kaz", "kbd", "kha", "khi", "khi", "khi", "khi", "khi", "khm", "khm", "kho", "kho", "kho", "kik", "kin", "kin", "kir", "kmb", "kmb", "kok", "kom", "kon", "kon", "kor", "kos", "kpe", "kpe", "kpe", "krc", "krl", "kro", "kro", "kro", "kro", "kro", "kro", "kro", "kru", "kua", "kua", "kum", "kur", "kut", "kut", "lad", "lad", "lad", "lah", "lah", "lah", "lam", "lao", "lao", "lat", "lav", "lav", "lav", "lez", "lim", "lin", "lit", "lol", "lol", "lol", "loz", "lsf", "ltz", "lua", "lub", "lub", "lub", "lug", "lug", "lui", "lun", "luo", "luo", "luo", "lus", "mac", "mad", "mag", "mag", "mag", "mag", "mah", "mai", "mak", "mal", "man", "man", "man", "man", "mao", "map", "map", "map", "map", "map", "map", "mar", "mar", "mas", "mas", "may", "mdf", "mdr", "men", "mga", "mga", "mga", "mga", "mic", "mic", "min", "min", "mis", "mis", "mkh", "mkh", "mlg", "mlt", "mnc", "mni", "mni", "mno", "mno", "moh", "mon", "mos", "mos", "mun", "mun", "mun", "mun", "mun", "mun", "mun", "mun", "mun", "mus", "mus", "mwl", "mwr", "myn", "myn", "myn", "myn", "myn", "myn", "myn", "myn", "myn", "myn", "myn", "myn", "myn", "myn", "myn", "myn", "myn", "myn", "myn", "myn", "myn", "myn", "myn", "myn", "myn", "myn", "myn", "myn", "myn", "myn", "myn", "myn", "myv", "nah", "nah", "nah", "nah", "nah", "nai", "nai", "nai", "nap", "nau", "nav", "nbl", "nde", "ndo", "ndo", "nds", "nds", "nds", "nds", "nep", "new", "new", "new", "nia", "nic", "nic", "nic", "nic", "nic", "niu", "niu", "nno", "nno", "nno", "nno", "nob", "nob", "nog", "nog", "non", "non", "non", "non", "non", "non", "non", "non", "nor", "nqo", "nso", "nso", "nso", "nub", "nub", "nub", "nub", "nub", "nub", "nwc", "nya", "nya", "nya", "nya", "nya", "nym", "nyn", "nyn", "nyo", "nzi", "nzi", "oci", "oci", "oci", "oci", "oci", "oci", "oci", "oci", "oci", "oci", "oci", "oci", "oji", "oji", "oji", "oji", "ori", "orm", "orm", "osa", "oss", "ota", "ota", "ota", "ota", "oto", "oto", "oto", "oto", "oto", "oto", "oto", "oto", "paa", "paa", "paa", "pag", "pal", "pal", "pal", "pal", "pal", "pam", "pan", "pan", "pan", "pap", "pau", "pau", "peo", "peo", "per", "per", "per", "per", "per", "phi", "phi", "phn", "pli", "pol", "pon", "por", "pra", "pra", "pro", "pro", "pro", "pus", "pus", "pus", "pus", "que", "que", "que", "raj", "rap", "rar", "rar", "roa", "roa", "roa", "roa", "roa", "roh", "roh", "roh", "rom", "rom", "rom", "rom", "rom", "rum", "rum", "run", "run", "run", "rup", "rup", "rus", "sad", "sag", "sah", "sah", "sah", "sai", "sai", "sal", "sal", "sal", "sal", "sal", "sal", "sal", "sal", "sal", "sal", "sal", "sal", "sal", "sal", "sal", "sal", "sal", "sam", "san", "san", "sas", "sat", "scn", "sco", "sco", "sel", "sel", "sem", "sem", "sem", "sem", "sem", "sem", "sem", "sem", "sem", "sem", "sem", "sga", "sga", "sga", "sgn", "shn", "shn", "sid", "sin", "sin", "sio", "sio", "sio", "sio", "sio", "sio", "sio", "sio", "sio", "sio", "sio", "sio", "sio", "sio", "sit", "sit", "sit", "sit", "sit", "sit", "sit", "sla", "sla", "sla", "sla", "sla", "sla", "sla", "sla", "sla", "sla", "sla", "sla", "sla", "sla", "slo", "slv", "sma", "sme", "smi", "smi", "smi", "smi", "smj", "smn", "smo", "sms", "sna", "sna", "snd", "snk", "sog", "som", "son", "son", "sot", "sot", "sot", "sot", "spa", "spa", "spdbnf", "spdbnf", "spdbnf", "srd", "srn", "srp", "srr", "srr", "ssa", "ssa", "ssa", "ssa", "ssa", "ssw", "ssw", "ssw", "suk", "sun", "sun", "sus", "sus", "sus", "sux", "swa", "swa", "swe", "swe", "syc", "syr", "syr", "tah", "tai", "tai", "tai", "tai", "tai", "tai", "tai", "tai", "tai", "tai", "tai", "tam", "tam", "tat", "tel", "tel", "tel", "tem", "tem", "tem", "ter", "ter", "tet", "tgk", "tgl", "tgl", "tha", "tha", "tib", "tig", "tir", "tir", "tiv", "tiv", "tiv", "tkl", "tlh", "tli", "tmh", "tmh", "tog", "tog", "ton", "tpi", "tsi", "tsi", "tsn", "tsn", "tso", "tuk", "tuk", "tum", "tup", "tup", "tur", "tut", "tut", "tut", "tut", "tut", "tut", "tvl", "twi", "twi", "tyv", "udm", "udm", "udm", "uga", "uga", "uig", "ukr", "umb", "umb", "umb", "urd", "urd", "uzb", "uzb", "vai", "vai", "vai", "ven", "vie", "vie", "vol", "vot", "wak", "wak", "wak", "wak", "wak", "wak", "wak", "wak", "wal", "wal", "wal", "war", "was", "wel", "wel", "wen", "wen", "wen", "wln", "wol", "wol", "wol", "xal", "xal", "xho", "xho", "xho", "yao", "yap", "yap", "yid", "yid", "yor", "ypk", "ypk", "zap", "zbl", "zbl", "zen", "zha", "zha", "znd", "znd", "zul", "zun", "zza", "zza", "zza", "zza", "zza", "zza"]

In [89]:
# Correspondances spécifiques validées avec la BML
CODES_LANGUE_BML = {"deu": "ger", "esp": "spa", "nld": "dut"}
CODES_PAYS_BML = {"DX": "de", "SP": "es", "XX": "xx", "YU": "yucs"}

# "xx" et "yucs" font partie des valeurs attendues dans le livrable,
# mais ne sont pas présents dans la liste de pays initiale.
CODES_PAYS_VALIDES = set(CODES_PAYS) | {"xx", "yucs"}
CODES_LANGUE_VALIDES = set(CODES_LANGUE) | {"und"}


In [90]:
CODES_FONCTION = {"040": "0990", # artiste : créateur / créatrice de l'œuvre
                  "650": "3250", # éditeur commercial (cartes postales)
                  "560": "0190", # commanditaire (affiches, photographies)
                  "570": "9990",
                  "070" : "0990",
                  "440" : "0440",
                  "260": "4210",
                  "390": "4010",
                  "320": "4040",
                  "340" : "9990",
                  "350": "0410",
                  "360" : "0410",
                  "530" : "0410",
                  "760" : "0410",
                  "510" : "0480",
                  "600" : "0530",
                  "610" : "3060",
                  "620" : "3060",
                  "740" : "2310",
                  "750": "2310",
                  "" : "" # avant on mettait 9990
                 }
                 
# Correspondances des codes fonctions 040, 650 et 560 validées par la BML


Référentiel catégorie technique : ici, on utilise les valeurs renseignées en 200\\$b (saisie manuelle) pour alimenter le référentiel "catégorie technique" de la BnF de façon minimale. Si la 200\\$b n'a pas été correctement saisie, cela risque de créer des anomalies

In [91]:
CATEGORIE_TECHNIQUE = {
    "Photographie": "photographie",
    "Carte postale": "procédé photomécanique",
    "Affiche": "procédé photomécanique",
    "Estampe": "estampe",
    "Manuscrit": "autre catégorie technique" # relevés d'inscriptions (monuments funéraires / architecturaux)
}

# Conversion après suppression éventuelle des crochets entourant 200$b
CATEGORIE_TECHNIQUE_NORMALISE = {k.casefold(): v for k, v in CATEGORIE_TECHNIQUE.items()}


### Création d'une classe records pour les notices unimarcs

#### Fonctions utilisées pour créer la classe records

In [92]:
def field_to_string(field):
    
    res = ""
    subfields = field.subfields
    for subfield in subfields:
        res += f"${subfield.code} {subfield.value} "
    
    return res

In [93]:
def note_exe2string(zone_note, string):
    for note in zone_note:
        for subfield in note.get_subfields("a"):
            string += subfield if subfield.endswith(".") else subfield + "."
           
    return string
    

#### Passer l'indexation Rameau en chaîne de caractères
Le but des deux fonctions suivantes est de permettre de passer d'une indexation Rameau en unimarc à des chaînes de caractères, qu'on chargera en indexation non contrôlée en Intermarc NG

In [94]:
def liste_subfields(field):
    liste_subfields = []
    for e in field.split("$"):
        if e:
            liste_subfields.append((e[0], e[1:].strip()))
    return liste_subfields

def marcfield2label(field):
    if field:
        repetitions = {"a": " -- ",
                    "x": " -- ",
                    "y": " -- ",
                    "l": " / ",
                    "d": " ; "}
        for char in repetitions:
            if re.search(f"\${char}([^$]+) \${char}", field) is not None:
                field = re.sub(f"\${char}([^$]+) \${char}", f"${char}\\1 {repetitions[char]}", field)
            if re.search(f"\${char}([^$]+) \${char}", field) is not None:
                field = re.sub(f"\${char}([^$]+) \${char}", f"${char}\\1 {repetitions[char]}", field)
        rules_multi = {"d k j": "(d-k-j)",
                       "f e": "f e"}  # date précise
        rules = {"a": "a",
                "b": ". b",
                "c": " c",
                "x": " -- x",
                "y": " -- y",
                "5": " -- 5",
                "i": " (i)",
                "k": "-k",
                "j": "-j",
                "d": " d",
                "h": " h",
                "t": " t",
                "l": " (l)",
                "m": ". m",
                "n": ". n",
                "z": " -- z",
                "f": " (f)",
                "v": " (v)"
        }
        for rule in rules_multi:
            chars = rules_multi[rule].split(" ")
            if len(chars) == 2:
                if re.search(f"\${chars[0]}([^$]+) \${chars[1]}([^$]+)", field) is not None:
                    field = re.sub(f"\${chars[0]}([^$]+) \${chars[1]}([^$]+)", r"(\1 ; \2)", field)
            if len(chars) == 3:
                if re.search(f"\${chars[0]}([^$]+) \${chars[1]}([^$]+) \${chars[2]}([^$]+)", field) is not None:
                    field = re.sub(f"\${chars[0]}([^$]+) \${chars[1]}([^$]+) \${chars[2]}([^$]+)( ?)", r"(\1-\2-\3)\4", field)
        field = liste_subfields(field)

        i = 0
        
        for subfield in field:
            try:
                rule = rules[subfield[0]]
            except KeyError:
                rule = f" -- {subfield[0]}"
            
            if ("(" in rule and "(" in subfield[1]) or (")" in rule and ")" in subfield[1]):
                rule = f" {subfield[0]}"
            # print(70, re.sub(f"\${char}([^$]+)", rule.replace(subfield[0], subfield[1]), subfield[1]))
            field[i] = rule.replace(subfield[0], subfield[1])
            
            i += 1
        field = "".join(field)
        field = field.replace("( ", "(").replace(" )", ")")
        field = field.replace(")(", ' ; ')
        field = field.replace(" ;; ", ' ; ').replace(" ; ; ", ' ; ')
        field = field.replace(";)", ')')
        field = field.replace("((", '(').replace("))", ")")
        while "  " in field:
            field = field.replace("  ", " ")
        if re.search("\$[\w]", field) is not None:
            field = re.sub("\$[\w]", " ¤¤ ", field)
    field = field.strip()
    
    return field

#### Classe Auteur
On utilise d'abord le mapping ISNI-NNA validé par la BML, puis le SRU de la BnF pour les ISNI absents du fichier ou sans NNA.
**Le format exact du NNA extrait du champ SRU `001` doit être validé avec le SRE / BnF.**


In [95]:
class Auteur:
    """A partir d'une zone unimarc de type 7XX, on définit les attributs à intégrer dans les colonnes Auteur de la matrice.
    On en fait un objet à part pour garder ensemble les informations liées à un même auteur."""

    def __init__(self, field):

        self.tag = field.tag if field else ""
        self.field = field if field else ""
        self.nom = next(iter(field.get_subfields("a")), "") if field else ""
        self.dol_o = next(iter(field.get_subfields("o")), "") if field else ""

        # Normalisation de l'ISNI et recherche du NNA (mapping local, puis SRU si nécessaire)
        self.isni = normaliser_isni(self.dol_o)
        self.nna = recuperer_nna(self.isni)

        # La matrice ne gère pas les codes fonction répétés, on ne prend donc que le premier
        self.fonction = next(iter(field.get_subfields("4")), "") if field else ""

        # Conversion des codes fonction Unimarc
        self.fonction = CODES_FONCTION.get(self.fonction, "")
        # Si un code fonction n'est pas dans la liste prévue, il devient indéterminé par défaut
        if self.fonction == "" and self.nom != "":
            self.fonction = "" # "9990" 
            # il n’est pas nécessaire de renseigner des valeurs par défaut pour les codes langue et les codes fonction 
            # lorsque ces informations sont absentes des données d’origine. 

        # Nature : Collectivité si 710/711/712 ; Personne si 700/701/702/720/721/722
        self.type = ""
        if self.tag in ["710", "711", "712"]:
            self.type = "Collectivité"
        elif self.tag in ["700", "701", "702", "720", "721", "722"]: # 721 & 722 sortis du chapeau, mais ce sont des auteurs théoriquement! (cf classe notice_unimarc)
            self.type = "Personne"

        self.prenom = ""
        if self.type == "Collectivité":
            self.prenom = next(iter(field.get_subfields("b")), "") if field else ""
        elif self.type == "Personne":
            self.prenom = next(iter(field.get_subfields("g")), "") if field else ""
            if self.prenom == "":
                self.prenom = next(iter(field.get_subfields("b")), "") if field else ""


#### Classe notice unimarc
Cette classe permet de constituer un objet avec les données à intégrer dans la matrice.
Les commentaires en français précisent les choix de conversion et les points restant à confirmer avec la BML / BnF.


In [ ]:
def insert (source_str, insert_str, pos):
    return source_str[:pos] + insert_str + source_str[pos:]


class notice_unimarc:
    
   
    def __init__(self, marc_record):
        self.marc_record = marc_record
        
        
       
        self.provenance = "BM de Lyon"
       
        self.num_notice = marc_record["001"].data
        self.ark = ""
        """ Attention ! Retier dièse sur la ligne suivante pour récupérer l'ark de la notice (absent de l'échantillon, mais présent dans le lot final)"""
        self.ark = marc_record["003"].data if marc_record["003"] else "" # Ça a changé ça, mais normalement pas de souci

        """Auteurs"""      
        champs_prioritaires = ["700", "710", "720", "701", "711", "721", "702", "712", "722"]
        auteurs = []
        for tag in champs_prioritaires:
            for field in marc_record.get_fields(tag):
                auteur = Auteur(field) if field else None
                if auteur and auteur not in auteurs and len(auteurs) <3:
                        auteurs.append(auteur)

        self.Auteur1 = auteurs[0] if len(auteurs)>0 else Auteur(None)
        self.Auteur2 = auteurs[1] if len(auteurs)>1 else Auteur(None)
        self.Auteur3 = auteurs[2] if len(auteurs)>2 else Auteur(None)
        
    
        
        
        """Langues"""
        self.liste_langues =  marc_record.get_fields("101")[0].get_subfields("a") if len(marc_record.get_fields("101"))>0 else []
        self.langue1 = self.liste_langues[0] if len(self.liste_langues) > 0 else None
        self.langue2 = self.liste_langues[1] if len(self.liste_langues) > 1 else None
        self.langue3 = self.liste_langues[2] if len(self.liste_langues) > 2 else None
        # Conversion des codes langue spécifiques à la BML (deu/esp/nld)
        self.langue1 = CODES_LANGUE_BML.get((self.langue1 or "").lower(), (self.langue1 or "").lower())
        self.langue2 = CODES_LANGUE_BML.get((self.langue2 or "").lower(), (self.langue2 or "").lower())
        self.langue3 = CODES_LANGUE_BML.get((self.langue3 or "").lower(), (self.langue3 or "").lower())
        if self.langue1 not in CODES_LANGUE_VALIDES:
            self.langue1 = "" # "und"
            # il n’est pas nécessaire de renseigner des valeurs par défaut pour les codes langue et les codes fonction 
            # lorsque ces informations sont absentes des données d’origine.
        if self.langue2 and self.langue2 not in CODES_LANGUE_VALIDES:
            self.langue2 = "" # "und"
            # il n’est pas nécessaire de renseigner des valeurs par défaut pour les codes langue et les codes fonction 
            # lorsque ces informations sont absentes des données d’origine.
        if self.langue3 and self.langue3 not in CODES_LANGUE_VALIDES:
            self.langue3 = "" # "und"
            # il n’est pas nécessaire de renseigner des valeurs par défaut pour les codes langue et les codes fonction 
            # lorsque ces informations sont absentes des données d’origine.

       

        
        """Attributs liés à publication et à l'impression
        Vu avec BmL : l'adresse bibliographique n'est pas systématiquement dans la même sous-zone, on en teste différentes"""
        ordre_adresse = [("210", "a"), ("210", "e"), ("214", "r")]
        self.lieu_edition = ""
        for subfield in ordre_adresse:
            if self.lieu_edition == "":
                self.lieu_edition = marc_record.get_fields(subfield[0])[0].get_subfields(subfield[1])[0] if len(marc_record.get_fields(subfield[0]))>0 and len(marc_record.get_fields(subfield[0])[0].get_subfields(subfield[1]))>0 else ""

       
        self.date_edition = "[s. d.]"
        self.editeur = "[s. n.]"
        self.lieu_impression = ""
        self.date_impression = ""

        if len(marc_record.get_fields("210"))>0 :
            # Valeurs par défaut "[s. d.]" / "[s. n.]" si la sous-zone est absente
            self.date_edition = marc_record.get_fields("210")[0].get_subfields("d")[0] if len(marc_record.get_fields("210")[0].get_subfields("d")) >0 else "[s. d.]"
            self.editeur = marc_record.get_fields("210")[0].get_subfields("c")[0] if len(marc_record.get_fields("210")[0].get_subfields("c"))>0 else "[s. n.]"
            self.lieu_impression = marc_record.get_fields("210")[0].get_subfields("e")[0] if len(marc_record.get_fields("210")[0].get_subfields("e"))>0 else ""
            self.date_impression = marc_record.get_fields("210")[0].get_subfields("g")[0] if len(marc_record.get_fields("210")[0].get_subfields("g"))>0 else ""

        
        """Date normalisée """
        self.date = marc_record.get_fields("100")[0].get_subfields("a")[0][9:17]if len(marc_record.get_fields("100"))>0 and len(marc_record.get_fields("100")[0].get_subfields("a"))>0   else ""
        regex_date = r"[0-9\.]{4}([0-9\. ]{4})?"
        search = re.fullmatch(regex_date, self.date)
        if search is None :
            self.date = re.sub(r"[^0-9\. ]", ".", self.date)
        # Un intervalle du type "1988...." indique une année connue, non la valeur "1"
        # VERIF QUE LE COMPORTEMENT SOIT BON, EN PARTICULIER CONCERNANT 
        # Les données manquantes doivent être remplacées par des points (par exemple 191.
        # cf mail de la semaine du 26
        if re.fullmatch(r"\d{4}\.{4}", self.date):
            self.date = self.date[:4]

        self.date = self.date.replace(" ", "")
        if self.date[0] == "'" or self.date[0] == "'": 
            self.date = self.date[1:]

        if len(self.date) == 8: 
            # ajouté le tiret du milieu
            self.date = insert (self.date, "-", 4)
            print(self.date)
        

        
        """Code pays"""
        self.pays = ""
        if len(marc_record.get_fields("102"))>0 :
            self.pays = marc_record.get_fields("102")[0].get_subfields("a")[0] if len(marc_record.get_fields("102")[0].get_subfields("a"))>0 else ""
        
        # Conversion des anciens codes pays BML (DX, SP, XX, YU)
        code_pays = self.pays.strip().upper()
        self.pays = CODES_PAYS_BML.get(code_pays, code_pays.lower())
        if self.pays not in CODES_PAYS_VALIDES:
            self.pays = "xx"
        # mais mais mais
        if NOM_FICHIER == 'GMB_Fichier5':
            self.pays = "fr"
       
        """Zone de titre"""
        self.titre = marc_record.get_fields("200")[0].get_subfields("a")[0] if len(marc_record.get_fields("200"))>0 and len(marc_record.get_fields("200")[0].get_subfields("a"))>0   else ""
        self.complement_titre = marc_record.get_fields("200")[0].get_subfields("e")[0] if len(marc_record.get_fields("200"))>0 and len(marc_record.get_fields("200")[0].get_subfields("e"))>0   else ""
        self.titre_parallele = marc_record.get_fields("200")[0].get_subfields("d")[0] if len(marc_record.get_fields("200"))>0 and len(marc_record.get_fields("200")[0].get_subfields("d"))>0   else ""
        self.mention_resp_1 = marc_record.get_fields("200")[0].get_subfields("f")[0] if len(marc_record.get_fields("200"))>0 and len(marc_record.get_fields("200")[0].get_subfields("f"))>0   else ""
        # I02-02 / I02-03 : 200$g, première et deuxième occurrence
        liste_200g = marc_record.get_fields("200")[0].get_subfields("g") if len(marc_record.get_fields("200"))>0 else []
        self.mention_resp_2 = liste_200g[0] if len(liste_200g) > 0 else ""
        self.mention_resp_3 = liste_200g[1] if len(liste_200g) > 1 else ""

        """Informations techniques"""
        self.tirage = marc_record.get_fields("205")[0].get_subfields("a")[0] if len(marc_record.get_fields("205"))>0 and len(marc_record.get_fields("205")[0].get_subfields("a"))>0   else ""      
        self.importance_materielle = marc_record.get_fields("215")[0].get_subfields("a")[0] if len(marc_record.get_fields("215"))>0 and len(marc_record.get_fields("215")[0].get_subfields("a"))>0   else ""
        self.couleur = marc_record.get_fields("215")[0].get_subfields("c")[0] if len(marc_record.get_fields("215"))>0 and len(marc_record.get_fields("215")[0].get_subfields("c"))>0   else "" 
        self.dimensions = marc_record.get_fields("215")[0].get_subfields("d")[0] if len(marc_record.get_fields("215"))>0 and len(marc_record.get_fields("215")[0].get_subfields("d"))>0   else ""
        # on supprime la mention cm (sans oublier de faire une conversion mm -> cm au besoin)
        # ON FAIT EGALEMENT LE CHOIX DE MODIFIER LA MENTION "(épr.)"
        self.dimensions = self.dimensions.replace("(épr.)", "")
        if "mm" in self.dimensions:
            self.dimensions = self.dimensions.replace("mm", "")
            # conversion
            # e.g. 36 x 24 mm converti en 
            tmp = self.dimensions.split("x")
            if len(tmp) == 2:
                # print(type(tmp[0]), type(tmp[1]))
                dim1, dim2 = float(tmp[0]) / 10, float(tmp[1]) / 10
                self.dimensions = f"{dim1} x {dim2}"
            else:
                print("*"*50)
                print("PROBLEME")
                print(self.dimensions)
        self.dimensions = self.dimensions.replace("cm", "")

        # 200$b est souvent saisi entre crochets ([Photographie], [Manuscrit], etc.)
        categorie = marc_record.get_fields("200")[0].get_subfields("b")[0] if len(marc_record.get_fields("200"))>0 and len(marc_record.get_fields("200")[0].get_subfields("b"))>0 else ""
        categorie = categorie.strip()
        if categorie.startswith("[") and categorie.endswith("]"):
            categorie = categorie[1:-1].strip()
        self.categorie_technique = CATEGORIE_TECHNIQUE_NORMALISE.get(categorie.casefold(), "autre catégorie technique")

        self.coordo_geographiques = marc_record.get_fields("206")[0].get_subfields("d")[0] if len(marc_record.get_fields("206"))>0 and len(marc_record.get_fields("206")[0].get_subfields("d"))>0   else ""     
     
                   

        
        """Indexation : on a deux systèmes différents selon que le document fait partie de Photographes en RA ou non"""
        self.indexation = []
        

        photographes_ra = False

        z461 =  marc_record.get_fields("461")
        for field in z461 :
            if  "Photographes en Rhône-Alpes" in next(iter(field.get_subfields("t")), ""):
                photographes_ra = True


        if photographes_ra :
            zones_indexation = [("608", 1), ("600", 3), ("606", 3)]
            for zone in zones_indexation: 
                    fields = marc_record.get_fields(zone[0])
                    i = 0
                    while i < zone[1] and i <len(fields):
                        
                        str_field = field_to_string(fields[i])
                        self.indexation.append(marcfield2label(str_field))
                        i+= 1
            liste_617 = fields = marc_record.get_fields("617")[:2]
            i = 0
            while len(self.indexation)<15 and i <len(liste_617):
                
                str_field = field_to_string(liste_617[i])
                self.indexation.append(marcfield2label(str_field))
                i += 1
                
            
        else : 
            # LA PARTIE MARRANTE, A BIEN VERIFIER!!!! 
            zones_indexation = [("608", 1), ("600", 3), ("607", 2), ("606", 5), ("610", 1)]
            
            for zone in zones_indexation: 
                fields = marc_record.get_fields(zone[0])
                i = 0
                while i < zone[1] and i <len(fields):
                    str_field = field_to_string(fields[i])
                    self.indexation.append(marcfield2label(str_field))
                    i+= 1
    
      
            for zone in zones_indexation:
                fields = marc_record.get_fields(zone[0])
                i = 0
                while  i <len(fields) and len(self.indexation)<15: 
                    str_field = field_to_string(fields[i])
                    str_field = marcfield2label(str_field)
                    if str_field not in self.indexation :
                        self.indexation.append(str_field)                     
                    i += 1
                        
           
        self.notes = ""
        zones_notes = ["300", "305", "307", "461", "488", "980"]

        # VERIF CE TRUC!!!
        # hmmm... CA M'EMBETE CE CODE!
        for zone in zones_notes:            
            fields = marc_record.get_fields(zone)                
            for field in fields:                    
                if zone == "300" :
                    note =  next(iter(field.get_subfields("a")), "")
                    if note != "":
                        self.notes += "Note générale : "
                        self.notes  += note if note.endswith(".") else note + ". "
                elif zone == "305":
                    note =  next(iter(field.get_subfields("a")), "")
                    if note != "":
                        self.notes += "Etats : "
                        self.notes  += note if note.endswith(".") else note + ". "
                elif zone == "307":
                    note =  next(iter(field.get_subfields("a")), "") 
                    if note != "":
                        self.notes += "Note sur les techniques : "
                        self.notes  += note if note.endswith(".") else note + ". "
                elif zone == "461":
                    note =  next(iter(field.get_subfields("t")), "")
                    if note != "":
                        self.notes += "Fait partie d'une série : "
                        self.notes  += note if note.endswith(".") else note + ". " 
                elif zone == "488":
                    note =  next(iter(field.get_subfields("t")), "") + " : " + next(iter(field.get_subfields("i")), "")
                    
                    if note != " : ":
                        self.notes += "En lien avec le document : "
                        self.notes  += note if note.endswith(".") else note + ". " 
                elif zone == "980":
                    note =  next(iter(field.get_subfields("a")), "")
                    if note != "":
                        self.notes += "Note de contexte historique : "
                        self.notes  += note if note.endswith(".") else note + ". " 

     

               
        """Notes d'exemplaire"""     
        self.note_exemplaires = ""
        self.z316 = marc_record.get_fields("316")
        self.z317 = marc_record.get_fields("317")
        self.note_exemplaires = note_exe2string(self.z316, self.note_exemplaires)
        self.note_exemplaires = note_exe2string(self.z317, self.note_exemplaires)
        # A confirmer avec la BML : 317$a contient aussi le fonds (I72-02).
        # On conserve la règle initiale dans I73-01 tant que la ventilation n'est pas validée.
        
        """Cote et code-barre
        Les champs 930$a et 017$a sont présents dans le lot définitif, mais peuvent manquer
        dans certaines notices. On vérifie leur présence avant d'en récupérer les valeurs."""
        self.cote = ""
       
        # En cas de doublon 930$a, la cote de l'original est en général sans préfixe NUM_.
        # Règle retenue ici à faire confirmer par la BML.
        cotes = [val.strip() for champ in marc_record.get_fields("930")
                 for val in champ.get_subfields("a") if val.strip()]
        self.cote = next((cote for cote in cotes if not cote.upper().startswith("NUM_")),
                         cotes[0] if cotes else "")
        self.code_barre = marc_record.get_fields("017")[0].get_subfields("a")[0] if len(marc_record.get_fields("017"))>0 and len(marc_record.get_fields("017")[0].get_subfields("a"))>0 else ""
        # print(marc_record.get_fields("017")[0].get_subfields("a")[0])
        # print(self.code_barre)
        """Codes projets
        On initie la liste avec le code projet générique, voir ici s'il faut en ajouter d'autres ?"""
        self.codes_projets = "LyoNu000" # ["LyoNu000"]

        self.code_projets2 = ""
        self.code_projets3 = "" # COMPRENDRE QUI METTRE DANS LE CODE_PROJET3 ou 2... POUR L'instant on met tout dans le 2
        # Dispatcher entre le 2 & le 3 ensuite!
        
        # Faisons juste une grosse disjonction de cas de fainéants...! 
        
    
    
            
    def get_subfield(self, field_tag, subfield_tag):
        #Récupère la valeur d'une sous-zone. Attention, ne fonctionne que si la zone et la sous-zone sont non répétables !
        record = self.marc_record
        field = record.get_fields(field_tag)[0] if record.get_fields(field_tag) else None
               
        res = ""
        if field is not None:
            
           
            res = field.get_subfields(subfield_tag)[0] if field.get_subfields(subfield_tag) else ""
        return res
        
    def __str__(self) :
        string = ""
        return string


## Création d'une liste de notices à partir du fichier ISO2709

In [97]:
list_indexation = []

In [98]:
list_records = []

In [99]:
#Changer ici le chemin du fichier avec le fichier définitif

# CORRECTIF ENCODAGE (ajouté) : les fichiers GMB sont en UTF-8, mais leur label (position 9) est vide,
# donc MARCReader(force_utf8=False) les décode en MARC-8 et produit des caractères erronés (é -> ©♭).
# Avec force_utf8=True, toutes les notices sont rejetées (None) à cause d'octets parasites non UTF-8
# (ex. \x03\xa2 en fin d'URL). On lit donc les notices une à une, on neutralise ces octets
# (remplacés par des espaces, sans changer la longueur) et on force le décodage UTF-8.
from pymarc import Record

def lire_records_utf8(fh):
    for chunk in fh.read().split(b'\x1d'):  # \x1d = fin de notice ISO 2709
        if len(chunk) < 24:
            continue
        buf = bytearray(chunk)
        while True:
            try:
                bytes(buf).decode('utf-8')
                break
            except UnicodeDecodeError as e:
                for k in range(e.start, e.end):
                    buf[k] = 0x20
        yield Record(data=bytes(buf) + b'\x1d', to_unicode=True, force_utf8=True)

# AJOUT : nom du fichier à convertir, réutilisé pour nommer le fichier Excel final
# (ex. 'GMB_Fichier2' -> GMB_Fichier2_partie_1.xlsx). Changer uniquement ici.
NOM_FICHIER = 'GMB_Fichier8'

# idx = 0

with open(NOM_FICHIER, 'rb') as fh:  # MODIF : 'GMB_Fichier2' remplacé par NOM_FICHIER
    reader = lire_records_utf8(fh)  # CORRECTIF : remplace MARCReader(fh, force_utf8=False)
    for record in tqdm(reader,desc="Processing Records"):
        notice = notice_unimarc(record)

        list_records.append(notice)

        list_indexation.extend(notice.indexation)   
        #idx += 1
        #if idx > 150: break
         


Processing Records: 0it [00:00, ?it/s]c:\Users\mathi\OneDrive\Bureau\Projet Numelyo\Projet Numelyo\SRUextractionMODIFIE.py:162: FutureWarning: This search incorrectly ignores the root element, and will be fixed in a future version.  If you rely on the current behaviour, change it to './/srw:diagnostics'
  if (self.result[0].find("//srw:diagnostics",
c:\Users\mathi\OneDrive\Bureau\Projet Numelyo\Projet Numelyo\SRUextractionMODIFIE.py:177: FutureWarning: This search incorrectly ignores the root element, and will be fixed in a future version.  If you rely on the current behaviour, change it to './/srw:numberOfRecords'
  if (self.result[0].find("//srw:numberOfRecords",
c:\Users\mathi\OneDrive\Bureau\Projet Numelyo\Projet Numelyo\SRUextractionMODIFIE.py:180: FutureWarning: This search incorrectly ignores the root element, and will be fixed in a future version.  If you rely on the current behaviour, change it to './/srw:numberOfRecords'
  self.nb_results = int(self.result[0].find("//srw:numb

1800-1899
1800-1899
1835-1836
1600-1699
1800-1899
1800-1899
1999-2000
1800-1899
1999-2000
1800-1899
1700-1799
1600-1699
1600-1699
1800-1899
1800-1899
1800-1899
1800-1899
1800-1899
1800-1899
1800-1899
1800-1899
1800-1899
1800-1899
1800-1899
1800-1899
1800-1899
1800-1899
1800-1899
1800-1899
1500-1599
1700-1799
1700-1799
1800-1899
1800-1899
1800-1899
1800-1899
1800-1899
1800-1899
1800-1899
1800-1899
1800-1899
1800-1899
1800-1899
1800-1899
1800-1899
1800-1899
1833-1865
1833-1865
1800-1899
1800-1899
1800-1899
1800-1899
1500-1599
1500-1599
1500-1599
1500-1599
1500-1599
1500-1599
1500-1599
1500-1599
1500-1599
1500-1599
1800-1899
1500-1599
1500-1599
1500-1599
1700-1799
1800-1899
1800-1899
1800-1899
1800-1899
1548-1555
1548-1555
1600-1699
1600-1699
1500-1599
1500-1599
1700-1799
1548-1555
1800-1899
1500-1599
1500-1599
1800-1899
1400-1599
1500-1599
1548-1555
1800-1899
1658-1659
1600-1699
1600-1699
1500-1599
1800-1899
1800-1899
1800-1899
1800-1899


Processing Records: 179it [00:01, 154.37it/s]

1800-1899
1800-1899
1800-1899
1500-1599
1548-1555
1548-1555
1600-1699
1800-1899
1683-1694
1800-1899
1800-1899
1500-1599
1500-1599
1500-1599
1800-1899
1700-1799
1600-1699
1600-1699
1600-1699
1800-1899
1800-1899
1800-1899
1800-1899
1700-1799
1700-1899
1548-1555
1800-1899
1548-1555
1600-1699
1800-1899
1500-1599
1548-1555
1548-1555
1548-1555
1800-1899
1800-1899
1600-1699
1800-1899
1800-1899
1800-1899
1548-1555
1600-1699
1600-1699
1600-1699
1600-1699
1800-1899
1754-1811
1754-1811
1600-1699
1600-1699
1548-1555
1600-1699
1800-1899
1700-1799
1600-1699
1658-1659
1800-1899
1800-1899
1800-1899
1800-1899
1800-1899


Processing Records: 266it [00:01, 195.76it/s]

1800-1899
1800-1899
1600-1699
1800-1899
1800-1899
1800-1899
1500-1599
1600-1699
1500-1599
1500-1599


... (98 outputs left out)

Processing Records: 9770it [00:20, 850.50it/s]

1800-1899
1500-1599
1800-1899
1500-1599
1500-1599
1500-1599
1700-1799
1500-1599
1800-1899
1800-1899
1800-1899
1800-1899
1800-1899
1800-1899
1800-1899
1800-1899
1800-1899
1800-1899
1800-1899
1800-1899
1500-1599
1800-1899
1800-1899
1600-1699
1600-1699
1800-1899
1800-1899
1800-1899
1800-1899
1800-1899
1800-1899
1800-1899
1800-1899
1800-1899
1800-1899
1700-1799
1700-1799
1548-1555
1500-1599
1500-1599
1500-1599
1500-1599
1500-1599
1500-1599
1500-1599
1500-1599
1600-1699
1500-1599
1500-1599
1500-1599
1500-1599
1600-1699
1572-1618
1500-1599
1745-1747
1600-1699
1700-1799
1700-1799
1700-1799


Processing Records: 9934it [00:20, 483.53it/s]

1600-1699
1600-1699
1600-1699
1700-1799
1700-1799
1700-1799
1700-1799
1500-1599
1500-1599
1800-1899
1800-1899
1800-1899
1800-1899
1500-1599
1600-1699
1530-1560
1548-1555
1548-1555


In [100]:
2130 + 517 + 1315 + 14470 + 12349 + 11562 + 8487 + 9934 + 67246 + 28

128038

In [101]:
print(len(list_records))

9934


In [102]:
counter = Counter(list_indexation)
subjects = [(count, subject) for subject, count in counter.items()]
df_subjects = pandas.DataFrame(subjects, columns = ["Count", "Subjects"])
df_subjects.to_excel("./test_indexation.xlsx")

## Création de la matrice

### Mapping
Concordance des informations entre les colonnes de la matrice et les attributs de l'objet notice unimarc

In [103]:
record2matrice = {
    "I01-01": "titre",
    "I01-02" : "complement_titre",
    "I01-03" : "titre_parallele",
    "I02-01" : "mention_resp_1",
    "I02-02" : "mention_resp_2",
    "I02-03" : "mention_resp_3",
    "I12-01" : "langue1", 
    "I12-02" : "langue2", 
    "I12-03" : "langue3",
    "I21-01" : "Auteur1.nna",
    "I21-02" : "Auteur1.nom",
    "I21-03" : "Auteur1.prenom",
    "I21-04" : "Auteur1.fonction",
    "I21-05" : "Auteur1.type",
    "I22-01" : "Auteur2.nna",
    "I22-02" : "Auteur2.nom",
    "I22-03" : "Auteur2.prenom",
    "I22-04": "Auteur2.fonction",
    "I22-05": "Auteur2.type",
    "I23-01" : "Auteur3.nna",
    "I23-02" : "Auteur3.nom",
    "I23-03" : "Auteur3.prenom",
    "I23-04": "Auteur3.fonction",
    "I23-05": "Auteur3.type",
    "I31-02" : "lieu_edition",
    "I32-01" : "date_edition",
    "I31-03" : "editeur",
    "I33-01" : "lieu_impression",
    "I33-03" : "date_impression",
    "I32-03" : "date",
    "I41-01" : "tirage",
    "I42-01": "importance_materielle",
    "I42-03" : "couleur", 
    "I42-04" : "dimensions",
    "I43-03" : "categorie_technique",
    "I46-02" : "coordo_geographiques",
    "I51-01" : "notes",
    "I11-01" : "pays",
    "I71-01" : "codes_projets",   
    # I71-02 / I71-03 : codes complémentaires à définir avec la BML
    # I71-02
    # I71-03
    # I72-02 : nom du fonds (317$a), à confirmer avec la BML
    "I72-01" : "provenance",
    "I72-03" : "cote",
    "I72-04" : "ark",
    "I72-05" : "code_barre",
    "I73-01" : "note_exemplaires"
   
}


### Liste des colonnes

In [104]:
COLS_MATRICE = ["I01-01", "I01-02", "I01-03", "I01-04", "I02-01", 
                "I02-02", "I02-03", "I11-01", "I12-01", "I12-02", 
                "I12-03", "I21-01", "I21-02", "I21-03", "I21-04", 
                "I21-05", "I22-01", "I22-02", "I22-03", "I22-04", 
                "I22-05", "I23-01", "I23-02", "I23-03", "I23-04", 
                "I23-05", "I31-02", "I31-03", "I32-01", "I32-03", 
                "I33-01", "I33-02", "I33-03", "I34-01", "I41-01", 
                "I42-01", "I42-03", "I42-04", "I43-01", "I43-02", 
                "I43-03", "I44-01", "I45-01", "I46-02", "I51-01", 
                "I52-01", "I71-01", "I71-02", "I71-03", "I72-01", 
                "I72-02", "I72-03", "I72-04", "I72-05", "I73-01", 
                "I73-02", "I73-03"]

len(COLS_MATRICE)


57

### Transformation des notices en matrice

In [105]:
def col2attr(key, record, dic) : 
    """Fonction qui permet d'aller récupérer les attributs à partir du dictionnaire ci-dessous"""
    if "." in dic[key] :
        el = dic[key].split(".")
        auteur = getattr(record, el[0])
        res = getattr(auteur, el[1])
    else : 
        res = getattr(record, dic[key])
    return res

In [106]:
records_data = []
for record in tqdm(list_records, desc="Processing Records"):
    d = {"id": record.num_notice}

    for col in COLS_MATRICE:
        if col in record2matrice:
            d[col] = col2attr(col, record, record2matrice)
        else:
            d[col] = ""

    # Indexations en colonnes supplémentaires : la nomenclature I61-00-XX
    # est conservée du script initial, à vérifier avec le format du livrable final.
    for i, item in enumerate(record.indexation):
        res = item.replace("-- frTICO", "").replace("-- rameau", "")
        # Supprimer les suffixes « -- frTico » et « -- rameau » ajoutés à la suite des sujets. 
        if len(res) >= 2 and res[-1] == " ": res = res[:-2]
        d[f"I61-00-{i:02d}"] = res

    records_data.append(d)

# Vérifier que toutes les correspondances sont présentes dans la matrice attendue
colonnes_non_declarees = set(record2matrice) - set(COLS_MATRICE)
if colonnes_non_declarees:
    raise ValueError(f"Colonnes absentes de COLS_MATRICE : {sorted(colonnes_non_declarees)}")

matrice = pandas.DataFrame(records_data)


Processing Records: 100%|██████████| 9934/9934 [00:00<00:00, 27924.66it/s]


In [107]:
records_data = []
for record in tqdm(list_records, desc="Processing Records"):
    d = {"id": record.num_notice}

    for col in COLS_MATRICE:
        if col in record2matrice:
            d[col] = col2attr(col, record, record2matrice)
        else:
            d[col] = ""

    # Indexations en colonnes supplémentaires : la nomenclature I61-00-XX
    # est conservée du script initial, à vérifier avec le format du livrable final.
    for i, item in enumerate(record.indexation):
        res = item.replace("-- frTICO", "").replace("-- rameau", "")
        # Supprimer les suffixes « -- frTico » et « -- rameau » ajoutés à la suite des sujets. 
        if len(res) >= 2 and res[-1] == " ": 
            res = res[:-2]

        if res != "" and (res[0] == "." or res[0] == "-"):
            res = res[2:] # PQ ça ne marche pas cette affaire ? I-61-02
            # res = res.lstrip("")
        # if "Rhône" in res: print(res)
        d[f"I61-00-{i:02d}"] = res
        # print(f"{(i+1):02d}")
        
        

Processing Records: 100%|██████████| 9934/9934 [00:00<00:00, 28977.29it/s]


In [108]:
# # Pour avoir les colonnes dans le bon ordre, sans perdre les colonnes non vides
# colonnes_indexation = sorted(col for col in matrice.columns if col.startswith("I61-00-"))
# colonnes_export = ["id"] + COLS_MATRICE + colonnes_indexation
# # La structure de base est conservée, même si certaines colonnes sont encore non renseignées
# df = matrice.reindex(columns=colonnes_export).fillna("")

# Récupérer et trier les colonnes d'indexation
colonnes_indexation = sorted(col for col in matrice.columns if col.startswith("I61-00-"))

# Construire la liste d'export en insérant les I61-00- juste avant la section I71
colonnes_export = ["id"]
indexation_inseree = False

for col in COLS_MATRICE:
    if not indexation_inseree and col.startswith("I71-"):
        colonnes_export.extend(colonnes_indexation)
        indexation_inseree = True
    colonnes_export.append(col)

# Sécurité : si aucune colonne I71 n'est trouvée dans COLS_MATRICE, on les ajoute à la fin
if not indexation_inseree:
    colonnes_export.extend(colonnes_indexation)

# Réindexation finale
df = matrice.reindex(columns=colonnes_export).fillna("")


### Export : à utiliser si petit fichier test

In [109]:
# df.to_excel("./matrice_lot_2.xlsx")

### Export : à utiliser pour + de 50 000 lignes

In [110]:
def clean_excel_illegal_chars(df):
    return df.map(
        lambda x: ILLEGAL_CHARACTERS_RE.sub("", str(x)) if isinstance(x, str) else x
    )

In [111]:
chunk_size = 50000

# Découpage en fichiers Excel
for i in range(0, len(df), chunk_size):
    chunk = df.iloc[i:i + chunk_size]
    chunk = clean_excel_illegal_chars(chunk) 
    chunk.to_excel(f"./RENDU_28_OCTOBRE_{NOM_FICHIER}_partie_{i//chunk_size + 1}.xlsx", index=False)  # MODIF : nom du fichier GMB dans le nom de sortie



### Contrôles après conversion
Vérifier notamment les NNA obtenus via SRU, les colonnes vides et le format attendu de la matrice. Les anomalies SRU restent consultables dans `ERREURS_SRU` ; les valeurs en 317$a et l'adresse bibliographique en 210/214 restent à valider avec la BML.


In [112]:
# Contrôles rapides à exécuter après la conversion
print("Nombre de notices :", len(df))
print("Correspondances ISNI/NNA dans le mapping :", len(ISNI_TO_NNA))
print("ISNI demandés au SRU :", len(SRU_CACHE_NNA))
print("NNA trouvés par le SRU :", sum(bool(nna) for nna in SRU_CACHE_NNA.values()))
print("Erreurs ou formats SRU à contrôler :", len(ERREURS_SRU))
print("Colonnes exportées :", len(df.columns))
# print(ERREURS_SRU)  # À utiliser ponctuellement pour le diagnostic


Nombre de notices : 9934
Correspondances ISNI/NNA dans le mapping : 1066
ISNI demandés au SRU : 61
NNA trouvés par le SRU : 0
Erreurs ou formats SRU à contrôler : 0
Colonnes exportées : 73
